# 22 · Check D3: the D2 check repeated on the A5 land maps

Pass marks fixed in `PLAN.md` (Check D3) before this notebook ran; the code is notebook 16's. The demo data now
holds the multinomial land maps chosen in A5 (`demo.land_method` in `configs/aether_us.yaml`). Answer key = the
main model's own plan run on the official maps (CDL, NWI, USPVDB) and NASA POWER on the demo's 90 m grid.

- **D3a:** precision ≥ 0.6 and F1 ≥ 0.5 for each of the three D2 questions.
- **D3b:** each answers in < 5 s including the language model, after one warm-up query (plans asked afresh,
  replies in `outputs/llm/d3/`).
- **Empty case:** the North Carolina heat question returns no places, naming the climate step and the 2-day cell.
- **Information:** the same scores for all ten page examples.

In [1]:
import json
import os
import time

import numpy as np
import pandas as pd
import torch

from askmap.aether_us import TextEncoder
from askmap.catalogue import catalogue_text
from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.demo.engine import Engine
from askmap.demo.serve import load_tiles, make_contexts, make_planner
from askmap.llm import make_client, system_prompt, translate
from askmap.plan import execute
from askmap.score import pooled_counts

load_env()
cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
AU = load_config(PROJECT_ROOT / "configs" / "aether_us.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
ROOT = data_root()
DEMO = ROOT / "demo"
RES = PROJECT_ROOT / "outputs" / "results"
meta = json.loads((DEMO / "meta.json").read_text())
LEAVES = cfg["classes"]["leaves"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
llm = cfg["llm"]
client = make_client(llm["base_url"])
system = system_prompt((PROJECT_ROOT / "configs" / "language.md").read_text(), catalogue_text(cfg))
T_START = time.time()
print("demo land method:", meta["land_method"], "| config:", AU["demo"]["land_method"])
if meta["land_method"] != AU["demo"]["land_method"]:
    raise RuntimeError("demo data were built with a different land method than the config names")


def build_engine(log_dir):
    def ask(question, path):
        return translate(client, llm["models"]["main"], system, question, path, llm)

    planner = make_planner(cfg, ask, log_dir)
    enc = TextEncoder(meta["encoder"], meta["encoder_kind"], device, os.environ["HF_HOME"])
    contexts = make_contexts(cfg, meta, DEMO)
    return Engine(
        load_tiles(DEMO, meta),
        LEAVES,
        enc,
        planner,
        contexts,
        meta["best"],
        CAP,
        top_share=meta["top_share"],
        not_ready=meta["not_ready"],
    )


LOG = PROJECT_ROOT / "outputs" / "llm" / "d3"
if LOG.exists() and any(LOG.iterdir()):
    raise RuntimeError(f"{LOG} already has replies; D3 must ask the language model afresh")
engine = build_engine(LOG)
t0 = time.time()
engine.search("hay fields", "north_dakota", "best")  # warm-up (not scored)
print(f"engine built and warmed up in {time.time() - t0:.0f}s; best combination = {engine.best}")

demo land method: context_more | config: context_more


/Volumes/disco2/github/ask-the-map-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1030 16:07:57.536000 28673 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 14407.21it/s]

engine built and warmed up in 1s; best combination = classifier


In [2]:
def score(q, area):
    t0 = time.perf_counter()
    r = engine.search(q, area, "best")
    seconds = time.perf_counter() - t0
    plan = r["plan"]
    tiles = engine.areas[area]
    pred = engine.answer_masks(plan, area)
    truth = execute(plan, {t.name: t.leaf.astype(np.int16) for t in tiles}, engine.contexts[area])
    c = pooled_counts(pred, truth, {t.name: t.valid for t in tiles})
    if c["predicted"] != r["facts"]["shown_pixels"]:
        raise AssertionError("answer masks and the shown places differ")
    ok = c["precision"] is not None and c["f1"] is not None and c["precision"] >= 0.6 and c["f1"] >= 0.5
    return {
        "question": q,
        "area": area,
        "seconds": round(seconds, 2),
        "plan_words": r["facts"]["plan_words"],
        **c,
        "a_mark": bool(ok),
        "b_mark": seconds < 5,
    }


D3 = [
    ("wetlands next to crop fields", "north_dakota"),
    ("corn fields where July 2021 had at least 5 days above 35 C", "north_dakota"),
    ("grassland near an existing solar farm", "north_carolina"),
]
rows = [score(q, a) for q, a in D3]
d3 = pd.DataFrame(rows)
d3[
    [
        "question",
        "area",
        "seconds",
        "predicted",
        "true",
        "both",
        "precision",
        "recall",
        "f1",
        "a_mark",
        "b_mark",
    ]
]

,question,area,seconds,predicted,true,both,precision,recall,f1,a_mark,b_mark
0,wetlands next to crop fields,north_dakota,1.23,4053,3148,1496,0.369109,0.475222,0.415498,False,True
1,corn fields where July 2021 had at least 5 day...,north_dakota,1.39,19172,18959,17775,0.927133,0.937549,0.932312,True,True
2,grassland near an existing solar farm,north_carolina,1.28,578,469,245,0.423875,0.522388,0.468004,False,True


In [3]:
for r in rows:
    print(f"{r['question']} ({r['area']}): {r['plan_words']}")

wetlands next to crop fields (north_dakota): Wetland, next to cropland.
corn fields where July 2021 had at least 5 days above 35 C (north_dakota): Corn, where July 2021 had at least 5 days with daily high temperature above 35 °C (NASA POWER).
grassland near an existing solar farm (north_carolina): Grassland, within 300 m of solar farm.


## The empty case: North Carolina, July 2021 heat

In [4]:
e = engine.search("corn fields where July 2021 had at least 5 days above 35 C", "north_carolina", "best")
reason = e["facts"]["empty_reason"] or ""
cells = e["facts"]["climate"][0]["cells"] if e["facts"]["climate"] else []
empty_ok = (
    e["facts"]["shown_pixels"] == 0
    and "No NASA POWER cell in this area meets" in reason
    and max(c["value"] for c in cells) == 2.0
    and "closest cell had 2" in reason
)
print(reason)
print("cell values:", [c["value"] for c in cells], "| empty-case message correct:", empty_ok)

No NASA POWER cell in this area meets "July 2021 had at least 5 days with daily high temperature above 35 °C": the closest cell had 2 against a threshold of 5 (days).
cell values: [0.0, 0.0, 2.0, 0.0] | empty-case message correct: True


## Information: all ten page examples

In [5]:
done = {(r["question"], r["area"]) for r in rows}
info = [
    score(x["question"], x["area"])
    for x in AU["demo"]["page_examples"]
    if (x["question"], x["area"]) not in done
]
allq = pd.DataFrame(rows + info)
allq[["question", "area", "seconds", "predicted", "true", "precision", "recall", "f1"]]

,question,area,seconds,predicted,true,precision,recall,f1
0,wetlands next to crop fields,north_dakota,1.23,4053,3148,0.369109,0.475222,0.415498
1,corn fields where July 2021 had at least 5 day...,north_dakota,1.39,19172,18959,0.927133,0.937549,0.932312
2,grassland near an existing solar farm,north_carolina,1.28,578,469,0.423875,0.522388,0.468004
3,Forest within 500 m of a solar farm,georgia,1.21,623,735,0.659711,0.559184,0.605302
4,Cropland that had an unusually wet spring in 2025,georgia,2.03,10679,10050,0.901863,0.958308,0.929230
5,Corn fields that had a very dry summer in 2022,north_dakota,1.94,4723,4449,0.899005,0.954372,0.925861
6,Corn fields that had an unusually wet spring i...,north_dakota,2.29,14743,14795,0.936716,0.933423,0.935067
7,Grassland and hay fields that had a very dry s...,north_dakota,2.07,22118,20091,0.838503,0.923100,0.878770
8,Grassland where the soil was unusually dry in ...,north_dakota,2.09,53756,48609,0.808059,0.893621,0.848689
9,Hay fields where the growing season of 2021 wa...,north_dakota,1.76,1952,2210,0.769980,0.680090,0.722249


In [6]:
d2 = json.loads((RES / "16_d2.json").read_text())
before = {(q["question"], q["area"]): q for q in d2["questions"]}
cmp = pd.DataFrame(
    [
        {
            "question": r["question"],
            "area": r["area"],
            "D2 precision": before[(r["question"], r["area"])]["precision"],
            "D3 precision": r["precision"],
            "D2 F1": before[(r["question"], r["area"])]["f1"],
            "D3 F1": r["f1"],
        }
        for r in rows
    ]
)
verdict = {
    "questions": rows,
    "page_examples": info,
    "D3a": bool(d3.a_mark.all()),
    "D3b": bool(d3.b_mark.all()),
    "empty_case_message": empty_ok,
    "land_method": meta["land_method"],
    "minutes": round((time.time() - T_START) / 60, 1),
}
(RES / "22_d3.json").write_text(json.dumps(verdict, indent=1, default=str))
print(
    f"D3a {'PASS' if verdict['D3a'] else 'FAIL'}; D3b {'PASS' if verdict['D3b'] else 'FAIL'}; "
    f"empty-case message {'OK' if empty_ok else 'WRONG'}; {verdict['minutes']} min"
)
cmp.round(3)

D3a FAIL; D3b PASS; empty-case message OK; 0.8 min


,question,area,D2 precision,D3 precision,D2 F1,D3 F1
0,wetlands next to crop fields,north_dakota,0.369,0.369,0.415,0.415
1,corn fields where July 2021 had at least 5 day...,north_dakota,0.927,0.927,0.932,0.932
2,grassland near an existing solar farm,north_carolina,0.424,0.424,0.468,0.468
